Naszym zadaniem w ramach konkursu DSC PJATK było przeprowadzenie eksploracyjnej analizy danych (EDA), zbudowanie modelu predykcyjnego cen samochodów na podstawie rzeczywistych danych z ofert sprzedaży oraz przedstawienie kluczowych wniosków. Dane zawierały zmienne takie jak marka, model, rok produkcji, przebieg, moc silnika czy emisja CO₂. Chcieliśmy nie tylko stworzyć działający model, ale też zrozumieć dane i opowiedzieć historię o tym, co wpływa na ceny samochodów.

Pierwszym krokiem było stworzenie wykresu rozkładu cen

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

TRAIN_FILE_NAME = "data/sales_ads_train.csv"

df = pd.read_csv(TRAIN_FILE_NAME)

df = df.dropna(subset=['Cena'])

df['Cena'] = pd.to_numeric(df['Cena'], errors='coerce')

max_reasonable_price = 1000000
df = df[(df['Cena'] >= 1000) & (df['Cena'] <= max_reasonable_price)]

outliers = df[df['Cena'] > max_reasonable_price]
if not outliers.empty:
    print("Znaleziono nierealistyczne ceny (powyżej 1,000,000 PLN):")
    print(outliers[['Marka_pojazdu', 'Model_pojazdu', 'Rok_produkcji', 'Cena']])

min_price = df['Cena'].min()
max_price = df['Cena'].max()

n = len(df)
k = int(1 + np.log2(n))

price_range = max_price - min_price
bin_width = price_range / k

round_base = 5000
bin_width = np.ceil(bin_width / round_base) * round_base

bins = np.arange(min_price, max_price + bin_width, bin_width)
bins = np.round(bins / round_base) * round_base

labels = [f"{int(bins[i])}-{int(bins[i+1])}" for i in range(len(bins)-1)]

df['Klasa_ceny'] = pd.cut(df['Cena'], bins=bins, labels=labels, include_lowest=True)

print("Liczba rekordów po filtracji:", n)
print("Liczba klas (reguła Sturge’a):", k)
print("Szerokość klasy:", bin_width)
print("\nLiczba samochodów w każdej klasie cenowej:")
print(df['Klasa_ceny'].value_counts().sort_index())

plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='Cena', bins=bins, kde=True, color='skyblue')

plt.title('Rozkład ceny samochodów z podziałem na klasy', fontsize=14)
plt.xlabel('Cena w PLN', fontsize=12)
plt.ylabel('Liczba samochodów', fontsize=12)

plt.xticks(bins, rotation=45)

plt.grid(True, alpha=0.3)
plt.show()

print("\nPodstawowe statystyki cen (w PLN):")
print(df['Cena'].describe())

Zauważyliśmy, że rozkład jest mocno prawoskośny – większość aut miała ceny w niższym zakresie, ale pojawiały się też oferty z "przesadzonymi" wartościami (np. miliony złotych). To sugerowało obecność odstających wartości (outliers), które mogły zaburzać model.
Powtórzyliśmy analizę, ale tym razem na skali logarytmicznej. Dzięki temu lepiej zobaczyliśmy rozproszenie cen i potwierdziliśmy, że mamy do czynienia z szerokim zakresem wartości – od tanich, używanych aut po drogie, premium modele

In [ ]:
import pandas as pd

TRAIN_FILE_NAME = "data/sales_ads_train.csv"

df = pd.read_csv(TRAIN_FILE_NAME)

missing_values = df.isnull().sum()

missing_values_df = pd.DataFrame(missing_values, columns=['Liczba brakujących wartości'])

missing_values_df.to_csv('missing_values_report.csv')

print("Liczba brakujących wartości:")
print(missing_values_df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

TRAIN_FILE_NAME = "data/sales_ads_train.csv"
df = pd.read_csv(TRAIN_FILE_NAME)

df = df.dropna(subset=['Cena'])
df['Cena'] = pd.to_numeric(df['Cena'], errors='coerce')

max_reasonable_price = 1_000_000  
df = df[(df['Cena'] >= 1_000) & (df['Cena'] <= max_reasonable_price)]

outliers = df[df['Cena'] > max_reasonable_price]
if not outliers.empty:
    print("Znaleziono nierealistyczne ceny (powyżej 1,000,000 PLN):")
    print(outliers[['Marka_pojazdu', 'Model_pojazdu', 'Rok_produkcji', 'Cena']])

n = len(df)
k = int(1 + np.log2(n))

log_bins = np.logspace(np.log10(df['Cena'].min()), np.log10(df['Cena'].max()), k)

plt.figure(figsize=(12, 6))
sns.histplot(df['Cena'], bins=log_bins, kde=True, color='skyblue')

plt.xscale('log')
plt.xlabel("Cena w PLN (log scale)")
plt.ylabel("Liczba samochodów")
plt.title("Rozkład ceny samochodów (skala logarytmiczna)")
plt.grid(True, which="both", linestyle="--", linewidth=0.5)

plt.xticks(log_bins, labels=[f"{int(x):,}" for x in log_bins], rotation=45)

plt.show()

print("\nPodstawowe statystyki cen (w PLN):")
print(df['Cena'].describe())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

TRAIN_FILE_NAME = "data/sales_ads_train.csv"

df = pd.read_csv(TRAIN_FILE_NAME)

df = df.dropna(subset=['Cena', 'Rok_produkcji', 'Przebieg_km', 'Moc_KM', 'Pojemnosc_cm3', 'Emisja_CO2', 'Liczba_drzwi'])

numeric_cols = ['Cena', 'Rok_produkcji', 'Przebieg_km', 'Moc_KM', 'Pojemnosc_cm3', 'Emisja_CO2', 'Liczba_drzwi']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

max_reasonable_price = 1000000
df = df[(df['Cena'] >= 1000) & (df['Cena'] <= max_reasonable_price)]

corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0, fmt='.2f')

plt.title('Mapa korelacji - interaktywny wykres z wartościami', fontsize=14)
plt.xlabel('Zmienne', fontsize=12)
plt.ylabel('Zmienne', fontsize=12)

plt.show()

Trafiliśmy na pierwsze poważne wyzwanie: brakujące dane, zwłaszcza w kolumnie "Stan" (nowy/używany). Sprawdziliśmy, jak często "Stan" jest pusty i w jakich kontekstach (np. w zależności od przebiegu – nowe auta miały zwykle niski przebieg).
Przeanalizowaliśmy też brakujące wartości dla wszystkich atrybutów, licząc ich liczbę i unikalne wartości. Np. "Waluta" brakowała w 3376 przypadkach, "Stan" w 3322, a "Emisja_CO2" aż w 75900 rekordach.

Benchmark: Pierwsza wersja modelu (v1)
W końcu stwierdziliśmy: "Dość błąkania się – robimy model na surowych danych i zobaczymy, co wyjdzie". Chcieliśmy mieć punkt odniesienia, czyli benchmark.
Wybraliśmy XGBoost, bo to jeden z najpotężniejszych algorytmów do regresji na danych strukturalnych. Jest znany z radzenia sobie z mieszanymi typami zmiennych (numeryczne + kategoryczne).
Stworzyliśmy pipeline:
Numeryczne: Imputacja medianą (bo mediana jest odporna na outliers).
Kategoryczne: Imputacja najczęściej występującą wartością + one-hot encoding (żeby model mógł je przetworzyć).
To standardowe podejście do danych mieszanych – proste, ale skuteczne na start.
Model i wyniki
Użyliśmy XGBoost z domyślnymi parametrami (n_estimators=100, learning_rate=0.1, max_depth=6).
Ocena: MAE, RMSE, R2 na zbiorze testowym. Dodatkowo wizualizowaliśmy feature importances, co pokazało, że "Moc_KM", "Pojemnosc_cm3" i "Marka_pojazdu" były kluczowe.
Po submit dostaliśmy wynik 0.0000 – coś poszło nie tak. Podejrzewaliśmy problem z formatem danych (np. float zamiast int).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt


print("1️⃣ Wczytuję dane z pliku CSV...")
dataset_df = pd.read_csv("data/sales_ads_train.csv")


kolumny_numeryczne = [
    "Rok_produkcji",
    "Przebieg_km",
    "Moc_KM",
    "Pojemnosc_cm3",
    "Emisja_CO2",
    "Liczba_drzwi"
]
kolumny_kategoryczne = [
    "Waluta",
    "Stan",
    "Marka_pojazdu",
    "Model_pojazdu",
    "Rodzaj_paliwa",
    "Typ_nadwozia",
    "Kolor",
    "Pierwszy_wlasciciel"
]

kolumny_numeryczne = [col for col in kolumny_numeryczne if col in dataset_df.columns]
kolumny_kategoryczne = [col for col in kolumny_kategoryczne if col in dataset_df.columns]
wybrane_kolumny = kolumny_numeryczne + kolumny_kategoryczne

X = dataset_df[wybrane_kolumny].copy()
y = dataset_df["Cena"].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, kolumny_numeryczne),
    ("cat", cat_pipeline, kolumny_kategoryczne)
])

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    objective="reg:squarederror",
    random_state=42,
    verbosity=1,
    eval_metric="rmse"  
)

print("🚀 Trenuję model XGBRegressor ...")
xgb_model.fit(
    X_train_transformed,
    y_train,
    eval_set=[(X_test_transformed, y_test)],
    verbose=True       
)


evals_result = xgb_model.evals_result()  
print("evals_result:", evals_result)

train_metrics = evals_result["validation_0"]["rmse"]
epochs = len(train_metrics)
x_axis = range(epochs)

plt.figure(figsize=(8, 5))
plt.plot(x_axis, train_metrics, label="RMSE (walidacja)")
plt.xlabel("Iteracja")
plt.ylabel("RMSE")
plt.title("Postęp trenowania modelu XGBoost")
plt.legend()
plt.show()

y_pred = xgb_model.predict(X_test_transformed)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print("MAE =", mae, "RMSE =", rmse, "R2 =", r2)

feature_importances = xgb_model.feature_importances_
feature_names = preprocessor.get_feature_names_out()
importance_df = pd.DataFrame({
    "Cechy": feature_names,
    "Ważność": feature_importances
}).sort_values(by="Ważność", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance_df["Cechy"][:20], importance_df["Ważność"][:20])
plt.gca().invert_yaxis()
plt.xlabel("Znaczenie cechy")
plt.ylabel("Cechy")
plt.title("Top 20 cech wg ważności w XGBoost")
plt.show()

importance_df.to_csv("feature_importance.csv", index=False)
print("✅ Zapisano feature_importance.csv")


print("📂 Wczytuję dane testowe...")
test_df = pd.read_csv("data/sales_ads_test.csv")

kolumny_numeryczne = [col for col in kolumny_numeryczne if col in test_df.columns]
kolumny_kategoryczne = [col for col in kolumny_kategoryczne if col in test_df.columns]
wybrane_kolumny = kolumny_numeryczne + kolumny_kategoryczne

X_submission = test_df[wybrane_kolumny].copy()

print("🔄 Przetwarzam zbiór testowy...")
X_submission_transformed = preprocessor.transform(X_submission)

print("🤖 Przewiduję ceny dla zbioru testowego...")
y_submission_pred = xgb_model.predict(X_submission_transformed)

submission_df = pd.DataFrame({
    "ID": test_df["ID"], 
    "Cena": y_submission_pred  
})

submission_df.to_csv("submission.csv", index=False)
print("✅ Zapisano submission.csv – gotowe do wysyłki!")



Iteracje: v2, v3 i v4
Wynik pierwszego submission wynosił około 25 tysięcy. Był to dobry wynik na początek ale wiedzieliśmy, że musimy go poprawić – zaczęliśmy iterować, by poprawić model i zmniejszyć błąd RMSE.
Wersja 2 (v2) - Przewidywane ceny zapisaliśmy jako int, a nie float, bo myśleliśmy, że to wymaganie konkursu.
Reszta bez zmian – ten sam preprocessing i model. Wynik nadal słaby, więc problem leżał gdzie indziej.
Wersja 3 (v3) - Zwiększyliśmy złożoność: Dostosowaliśmy hiperparametry (np. więcej drzew, głębsze drzewa) i dodaliśmy walidację krzyżową (CV=5), żeby lepiej ocenić generalizację. CV pozwala uniknąć overfittingu i daje stabilniejszą ocenę modelu.
Wersja 4 (v4) – Weszliśmy na wyższy poziom: Użyliśmy Optuny do optymalizacji hiperparametrów (np. n_estimators=3000, learning_rate=0.0218, max_depth=11). Zastąpiliśmy one-hot encoding target encodingiem, bo lepiej radzi sobie z licznymi kategoriami. Finalnie wersja 4 pozwoliła nam zniżyć się do wyniku RMSE 21 tysięcy.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import optuna

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from category_encoders import TargetEncoder

print("Wczytuję dane z pliku CSV (train)...")
df = pd.read_csv("data/sales_ads_train.csv")

num_cols = ["Rok_produkcji", "Przebieg_km", "Moc_KM", "Pojemnosc_cm3", "Emisja_CO2", "Liczba_drzwi"]
cat_cols = ["Waluta", "Stan", "Marka_pojazdu", "Model_pojazdu", "Rodzaj_paliwa", "Typ_nadwozia", "Kolor", "Pierwszy_wlasciciel"]
num_cols = [col for col in num_cols if col in df.columns]
cat_cols = [col for col in cat_cols if col in df.columns]
all_cols = num_cols + cat_cols

X = df[all_cols].copy()
y = df["Cena"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", TargetEncoder())
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

X_train_transformed = preprocessor.fit_transform(X_train, y_train)
X_test_transformed = preprocessor.transform(X_test)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 1000, 10000, step=1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),
        "random_state": 42,
        "tree_method": "gpu_hist" if xgb.__version__ >= "1.0.0" else "hist",
    }

    model = xgb.XGBRegressor(**params)
    scores = cross_val_score(model, X_train_transformed, y_train, scoring="neg_root_mean_squared_error", cv=5)
    return -scores.mean()

print("Optymalizacja hiperparametrów za pomocą Optuny...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print("Najlepsze parametry znalezione przez Optunę:", best_params)

xgb_model = xgb.XGBRegressor(**best_params)

print("Przeprowadzam walidację krzyżową dla finalnego modelu (CV=5)...")
cv_scores = cross_val_score(xgb_model, X_train_transformed, y_train, scoring="neg_root_mean_squared_error", cv=5)
mean_cv_rmse = -cv_scores.mean()
print(f"Średni RMSE w walidacji krzyżowej: {mean_cv_rmse:.2f}")

print("Trenuję model XGBRegressor...")
xgb_model.fit(
    X_train_transformed,
    y_train,
    eval_set=[(X_test_transformed, y_test)],
    early_stopping_rounds=100,
    verbose=True
)

y_pred = xgb_model.predict(X_test_transformed)
mae_value = mean_absolute_error(y_test, y_pred)
rmse_value = np.sqrt(mean_squared_error(y_test, y_pred))
r2_value = r2_score(y_test, y_pred)

print(f"MAE = {mae_value:.3f}, RMSE = {rmse_value:.3f}, R2 = {r2_value:.3f}")

importance = xgb_model.get_booster().get_score(importance_type="gain")
importance_df = pd.DataFrame({
    "Cecha": list(importance.keys()),
    "Waga": list(importance.values())
}).sort_values(by="Waga", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance_df["Cecha"][:20], importance_df["Waga"][:20])
plt.gca().invert_yaxis()
plt.xlabel("Ważność")
plt.ylabel("Cecha")
plt.title("Top 20 cech wg ważności w XGBoost")
plt.show()

importance_df.to_csv("feature_importance.csv", index=False)
print("Zapisano feature_importance.csv")

print("Wczytuję dane testowe (sales_ads_test.csv)...")
test_df = pd.read_csv("data/sales_ads_test.csv")

test_num_cols = [col for col in num_cols if col in test_df.columns]
test_cat_cols = [col for col in cat_cols if col in test_df.columns]
test_all_cols = test_num_cols + test_cat_cols

X_submission = test_df[test_all_cols].copy()

print("Przetwarzam zbiór testowy (transformuję) ...")
X_submission_transformed = preprocessor.transform(X_submission)

print("Przewiduję ceny...")
y_submission_pred = xgb_model.predict(X_submission_transformed)

y_submission_pred = np.round(y_submission_pred).astype(int)

submission_df = pd.DataFrame({
    "ID": test_df["ID"],
    "Cena": y_submission_pred
})

submission_df.to_csv("submission.csv", index=False)
print("Zapisano submission.csv – plik gotowy do wysyłki.")

 Mimo postępów w modelowaniu, dane nadal były "brudne". Postanowiliśmy więc odrzucić zbędne kolumny oraz usunąć wiersze które były niejednoznaczne i ciężkie do imputacji.
 Kolumny "Emisja_CO2", "Liczba_drzwi", "Kolor", "Lokalizacja_oferty" – miały mały wpływ na cenę (potwierdzone analizą korelacji i feature importances)."Kraj_pochodzenia" też odrzuciliśmy – analiza dla Audi i Toyoty pokazała brak wpływu.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import glob
import os

csv_files = glob.glob("data/origin/price_correlation_by_country_*.csv")
latest_file = max(csv_files, key=os.path.getctime)
df = pd.read_csv(latest_file)

audi_df = df[df["Marka"] == "Audi"]

top_5_audi = audi_df.nlargest(5, "Liczba_wystapien")

top_5_audi["Grupa"] = top_5_audi.apply(
    lambda row: f"{row['Marka']} {row['Model']} {row['Rok_produkcji']} {row['Rodzaj_paliwa']} poj. {row['Pojemnosc']} (liczność: {row['Liczba_wystapien']})",
    axis=1
)

plt.figure(figsize=(14, 6))
bars = plt.bar(top_5_audi["Grupa"], top_5_audi["Korelacja_Cena_Kraj"], color="skyblue", edgecolor="black")

plt.xlabel("Grupa pojazdów")
plt.ylabel("Korelacja między ceną a krajem pochodzenia")
plt.title("Korelacja ceny z krajem pochodzenia dla 5 najliczniejszych grup Audi")
plt.xticks(rotation=45, ha="right")
plt.grid(True, axis="y", linestyle="--", alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, f"{yval:.2f}", ha="center", va="bottom" if yval >= 0 else "top")

plt.tight_layout()

plt.show()

W międzyczasie zauważyliśmy, że samochody elektryczne oraz hybrydowe, nawet w obrębie jednego modelu, były droższe od swoich spalinowych rówieśników. Sworzyliśmy wykres przedstawiający różnice śrendniej ceny. Dla modeli które mają reprezentantów w każdej z trzech grup napędowych. Niestety badanie, mimo, że potwierdził naszą tezę, nie wniosło poprawy do wyniku. Dlaczego? otóż ceny dla pojazdów spalinowych były mocno zaniżone, ponieważ jest dużo aut "starszych" z niską ceną które sa z tego samego modelu, np. z lat 90, kiedy hybrydy a tym bardziej elektryczne napędy nie były dostępne. Sprawdziliśmy w kodzie jaki rok jest "optymalny" i stanowiłby dobrą granice porówniania. Zrezygnowaliśmy z tego badania ponieważ pomimo większego grzebania w kodzie, nie zauważyliśmy poprawy w wynikach.

In [ ]:
import pandas as pd
import numpy as np

def find_optimal_production_year(df, min_sample_size=50, max_age=20):
    df_clean = df.dropna(subset=["Rok_produkcji", "Cena"])
    
    df_clean["Rok_produkcji"] = df_clean["Rok_produkcji"].astype(int)
    
    current_year = 2021
    
    best_year = None
    min_std_dev = float('inf')
    valid_years = []
    
    for start_year in range(current_year - max_age, current_year):
        temp_df = df_clean[df_clean["Rok_produkcji"] >= start_year]
        
        if len(temp_df) < min_sample_size:
            continue
        
        std_dev = temp_df["Cena"].std()
        
        valid_years.append((start_year, len(temp_df), std_dev))
        
        if std_dev < min_std_dev:
            min_std_dev = std_dev
            best_year = start_year
    
    return best_year, valid_years


import pandas as pd
import matplotlib.pyplot as plt

TRAIN_FILE_NAME = "data/sales_ads_train.csv"
df = pd.read_csv(TRAIN_FILE_NAME)

df = df.dropna(subset=["Model_pojazdu", "Rodzaj_paliwa", "Cena", "Rok_produkcji"])

paliwa = {
    "Spalinowe": ["Gasoline", "Diesel"],
    "Hybrydowe": ["Hybrid"],
    "Elektryczne": ["Electric"]
}

df["Paliwo_typ"] = df["Rodzaj_paliwa"].apply(
    lambda x: "Spalinowe" if x in paliwa["Spalinowe"] else "Hybrydowe" if x in paliwa["Hybrydowe"] else "Elektryczne" if x in paliwa["Elektryczne"] else "Inne"
)

model_fuel_counts = df.groupby(["Model_pojazdu", "Paliwo_typ"]).size().unstack(fill_value=0)

models_with_spalinowe = model_fuel_counts[model_fuel_counts["Spalinowe"] > 0].index
models_with_elektryczne_or_hybrydowe = model_fuel_counts[(model_fuel_counts["Elektryczne"] > 0) | (model_fuel_counts["Hybrydowe"] > 0)].index
models_filtered = models_with_spalinowe.intersection(models_with_elektryczne_or_hybrydowe)

filtered_df = df[df["Model_pojazdu"].isin(models_filtered)]

optimal_year, _ = find_optimal_production_year(filtered_df)
filtered_df = filtered_df[filtered_df["Rok_produkcji"] >= optimal_year]

price_summary = filtered_df.groupby(["Model_pojazdu", "Paliwo_typ"])["Cena"].mean().reset_index()

top_10_models = filtered_df["Model_pojazdu"].value_counts().head(10).index
top_10_df = price_summary[price_summary["Model_pojazdu"].isin(top_10_models)]

pivot_data = top_10_df.pivot(index="Model_pojazdu", columns="Paliwo_typ", values="Cena").fillna(0)

plt.figure(figsize=(14, 8))
pivot_data[["Spalinowe", "Hybrydowe", "Elektryczne"]].plot(kind="bar", width=0.8, colormap="viridis")

plt.title("Średnia cena dla modeli z wersją spalinową oraz hybrydową/elektryczną (od roku {})".format(optimal_year), fontsize=16)
plt.xlabel("Model pojazdu", fontsize=12)
plt.ylabel("Średnia cena", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Typ paliwa")
plt.grid(True, axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

W temacie kodu, postanowliśmy, aby "oczyszczone" dane wykorzystać w naszym kodzie. W wersji 5 poprawiliśmy model, zmieniając źródło danych na oczyszczony zbiór z bardziej istotnymi cechami, takimi jak Wiek_pojazdu czy Generacja_pojazdu, oraz zwiększając liczbę prób w Optunie z 20 do 50, co pozwoliło lepiej dostroić hiperparametry. Naszą główną motywacją było obniżenie RMSE i poprawa predykcji cen poprzez wykorzystanie bardziej szczegółowych danych i dokładniejszą optymalizację modelu. Usunęliśmy mniej istotne cechy, jak Emisja_CO2 czy Kolor, aby skupić się na zmiennych silniej skorelowanych z ceną. Postanowiliśmy usprawnić preprocessing, dodać analizę reszt, zapisać najlepsze parametry oraz rozważyć skalowanie zmiennych numerycznych dla jeszcze lepszych wyników. Niestety wynik nie poprawił się, przetrenowaliśmy model i niepoprawnie przeprowadziliśmy czyszczenie danych. Zrezygnowaliśmy też z osobnych plików do czyszczenia danych i postanowliśmy stworzyć bardziej modularny projekt w jednym pliku w którym uwzględniamy już czyszczenie, trenowanie modelu oraz predykcje.

In [ ]:

import os
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from category_encoders import TargetEncoder

EXCHANGE_RATE_EUR_TO_PLN = 4.5 

class DataLoader:
    def __init__(self, train_path, test_path):
        self.train_path = train_path
        self.test_path = test_path
    
    def load_data(self):
        print("Wczytuję dane z pliku CSV (train)...")
        df = pd.read_csv(self.train_path)
        df = self.convert_prices_to_pln(df)
        return df
    
    def load_test_data(self):
        print("Wczytuję dane testowe...")
        df = pd.read_csv(self.test_path)
        df["Is_EUR"] = df["Waluta"] == "EUR" 
        return df

    @staticmethod
    def convert_prices_to_pln(df):
        """ Konwertuje ceny w EUR na PLN w zbiorze treningowym """
        df.loc[df["Waluta"] == "EUR", "Cena"] *= EXCHANGE_RATE_EUR_TO_PLN
        df["Waluta"] = "PLN" 
        return df

class Preprocessor:
    def __init__(self, df):
        self.num_cols = ["Rok_produkcji", "Przebieg_km", "Moc_KM", "Pojemnosc_cm3", "Emisja_CO2", "Liczba_drzwi"]
        self.cat_cols = ["Waluta", "Stan", "Marka_pojazdu", "Model_pojazdu", "Rodzaj_paliwa", "Typ_nadwozia", "Kolor", "Pierwszy_wlasciciel"]
        self.num_cols = [col for col in self.num_cols if col in df.columns]
        self.cat_cols = [col for col in self.cat_cols if col in df.columns]
        
        self.num_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])
        self.cat_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", TargetEncoder())
        ])
        self.preprocessor = ColumnTransformer([
            ("num", self.num_pipeline, self.num_cols),
            ("cat", self.cat_pipeline, self.cat_cols)
        ])
    
    def transform(self, X, y=None):
        if y is not None:
            return self.preprocessor.fit_transform(X, y)
        return self.preprocessor.transform(X)

class ModelTrainer:
    def __init__(self, params):
        self.model = xgb.XGBRegressor(**params)
    
    def cross_validate(self, X, y):
        print("Przeprowadzam walidację krzyżową...")
        cv_scores = cross_val_score(self.model, X, y, scoring="neg_root_mean_squared_error", cv=5)
        print(f"Średni RMSE w walidacji krzyżowej: {-cv_scores.mean():.2f}")
    
    def train(self, X, y):
        print("Trenuję model XGBRegressor...")
        self.model.fit(X, y)
    
    def evaluate(self, X, y):
        y_pred = self.model.predict(X)
        print(f"MAE = {mean_absolute_error(y, y_pred):.3f}, RMSE = {np.sqrt(mean_squared_error(y, y_pred)):.3f}, R2 = {r2_score(y, y_pred):.3f}")
    
    def predict(self, X):
        return np.round(self.model.predict(X)).astype(int)
    
    def feature_importance(self):
        importance = self.model.get_booster().get_score(importance_type="gain")
        return pd.DataFrame({"Cecha": list(importance.keys()), "Waga": list(importance.values())}).sort_values(by="Waga", ascending=False)

class SubmissionHandler:
    def __init__(self, output_folder="tester"):
        os.makedirs(output_folder, exist_ok=True)
        self.output_folder = output_folder
    
    def save_submission(self, test_df, predictions, filename="submission.csv"):
        test_df["Cena"] = predictions

        test_df.loc[test_df["Is_EUR"], "Cena"] /= EXCHANGE_RATE_EUR_TO_PLN
        test_df["Cena"] = test_df["Cena"].round(2) 

        submission_df = test_df[["ID", "Cena"]]
        path = os.path.join(self.output_folder, filename)
        submission_df.to_csv(path, index=False)
        print(f"Zapisano {path} – plik gotowy do wysyłki.")

if __name__ == "__main__":
    data_loader = DataLoader("data/sales_ads_train.csv", "data/sales_ads_test.csv")
    df = data_loader.load_data()
    
    preprocessor = Preprocessor(df)
    X = df[preprocessor.num_cols + preprocessor.cat_cols].copy()
    y = df["Cena"].copy()
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    X_train_transformed = preprocessor.transform(X_train, y_train)
    X_test_transformed = preprocessor.transform(X_test)
    
    best_params = {
        'n_estimators': 3000,
        'learning_rate': 0.0218,
        'max_depth': 11,
        'min_child_weight': 3,
        'subsample': 0.717,
        'colsample_bytree': 0.615,
        'reg_alpha': 0.134,
        'reg_lambda': 7.813,
        'random_state': 42,
        'tree_method': 'gpu_hist' if xgb.__version__ >= "1.0.0" else "hist"
    }
    
    model_trainer = ModelTrainer(best_params)
    model_trainer.cross_validate(X_train_transformed, y_train)
    model_trainer.train(X_train_transformed, y_train)
    model_trainer.evaluate(X_test_transformed, y_test)
    
    feature_importance_df = model_trainer.feature_importance()
    feature_importance_df.to_csv("feature_importance.csv", index=False)
    
    test_df = data_loader.load_test_data()
    X_submission = test_df[preprocessor.num_cols + preprocessor.cat_cols].copy()
    X_submission_transformed = preprocessor.transform(X_submission)
    y_submission_pred = model_trainer.predict(X_submission_transformed)
    
    submission_handler = SubmissionHandler()
    submission_handler.save_submission(test_df, y_submission_pred)

W kolejnej wersji w porównaniu dopoprzedniej przywróciliśmy optymalizację hiperparametrów za pomocą Optuny (20 prób), aby dynamicznie dostosować parametry modelu do danych, co miało na celu obniżenie RMSE i poprawę predykcji cen. Wprowadziliśmy obsługę GPU poprzez parametry device="cuda", tree_method="hist" i predictor="gpu_predictor", zwiększając wydajność treningu i predykcji na dużych zbiorach danych. Usprawniliśmy preprocessing, zmieniając strategię imputacji zmiennych kategorycznych na strategy="constant", fill_value="missing" oraz dodając wstępne wypełnianie brakujących wartości w kluczowych kolumnach (Marka_pojazdu, Model_pojazdu itp.), co zapewnia większą kontrolę nad danymi i ich spójność. Postanowiliśmy zoptymalizować wydajność obliczeń na GPU, poprawić obsługę brakujących danych, dodać analizę reszt oraz rozważyć skalowanie zmiennych numerycznych dla dalszej poprawy wyników.

In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import optuna

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from category_encoders import TargetEncoder

EXCHANGE_RATE_EUR_TO_PLN = 4.5 

class DataLoader:
    def __init__(self, train_path, test_path):
        self.train_path = train_path
        self.test_path = test_path

    def load_data(self):
        print("Wczytuję dane z pliku CSV (train)...")
        df = pd.read_csv(self.train_path)

        df["Cena"] = df["Cena"].astype(float)

        df = self.convert_prices_to_pln(df)

        for col in ["Marka_pojazdu", "Model_pojazdu", "Wersja_pojazdu", "Generacja_pojazdu"]:
            if col in df.columns:
                df[col] = df[col].fillna("missing")
        
        return df

    def load_test_data(self):
        print("Wczytuję dane testowe...")
        df = pd.read_csv(self.test_path)
        df["Is_EUR"] = df["Waluta"] == "EUR" 

     
        for col in ["Marka_pojazdu", "Model_pojazdu", "Wersja_pojazdu", "Generacja_pojazdu"]:
            if col in df.columns:
                df[col] = df[col].fillna("missing")

        return df

    @staticmethod
    def convert_prices_to_pln(df):
        """Konwertuje ceny w EUR na PLN w zbiorze treningowym"""
        mask_eur = df["Waluta"] == "EUR"
        df["Cena"] = df["Cena"].astype(float)
        df.loc[mask_eur, "Cena"] = df.loc[mask_eur, "Cena"] * EXCHANGE_RATE_EUR_TO_PLN
        df["Waluta"] = "PLN" 
        return df

class Preprocessor:
    def __init__(self, df):
        self.num_cols = [
            "Rok_produkcji",
            "Przebieg_km",
            "Moc_KM",
            "Pojemnosc_cm3",
            "Emisja_CO2",
            "Liczba_drzwi"
        ]
        
        self.cat_cols = [
            "Waluta",
            "Stan",
            "Marka_pojazdu",
            "Model_pojazdu",
            "Rodzaj_paliwa",
            "Typ_nadwozia",
            "Kolor",
            "Pierwszy_wlasciciel",
            "Wersja_pojazdu",
            "Generacja_pojazdu"
        ]
        
        self.num_cols = [c for c in self.num_cols if c in df.columns]
        self.cat_cols = [c for c in self.cat_cols if c in df.columns]

        self.num_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])
        
        self.cat_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", TargetEncoder())  
        ])
        
        self.preprocessor = ColumnTransformer([
            ("num", self.num_pipeline, self.num_cols),
            ("cat", self.cat_pipeline, self.cat_cols)
        ])

    def transform(self, X, y=None):
        if y is not None:
            return self.preprocessor.fit_transform(X, y)
        return self.preprocessor.transform(X)

class ModelTrainer:
    """ Pomocnicza klasa do trenowania i ewaluacji modelu XGB. """
    def __init__(self, params):
        self.model = xgb.XGBRegressor(**params)
    
    def cross_validate(self, X, y):
        print("Przeprowadzam walidację krzyżową...")
        cv_scores = cross_val_score(
            self.model, X, y,
            scoring="neg_root_mean_squared_error",
            cv=5
        )
        print(f"Średni RMSE w walidacji krzyżowej: {-cv_scores.mean():.2f}")
    
    def train(self, X, y):
        print("Trenuję model XGBRegressor...")
        self.model.fit(X, y)
    
    def evaluate(self, X, y):
        y_pred = self.model.predict(X)
        mae = mean_absolute_error(y, y_pred)
        rmse = np.sqrt(mean_squared_error(y, y_pred))
        r2 = r2_score(y, y_pred)
        print(f"MAE = {mae:.3f}, RMSE = {rmse:.3f}, R2 = {r2:.3f}")
    
    def predict(self, X):
        return np.round(self.model.predict(X)).astype(int)
    
    def feature_importance(self):
        booster = self.model.get_booster()
        importance = booster.get_score(importance_type="gain")
        df_imp = pd.DataFrame({
            "Cecha": list(importance.keys()),
            "Waga": list(importance.values())
        })
        return df_imp.sort_values(by="Waga", ascending=False)

class SubmissionHandler:
    """ Zapisywanie predykcji do pliku CSV. """
    def __init__(self, output_folder="tester"):
        os.makedirs(output_folder, exist_ok=True)
        self.output_folder = output_folder
    
    def save_submission(self, test_df, predictions, filename="submission.csv"):
        test_df["Cena"] = predictions
        
        mask_eur = test_df["Is_EUR"]
        test_df.loc[mask_eur, "Cena"] = test_df.loc[mask_eur, "Cena"] / EXCHANGE_RATE_EUR_TO_PLN
        
        test_df["Cena"] = test_df["Cena"].round(2)
        
        submission_df = test_df[["ID", "Cena"]]
        path = os.path.join(self.output_folder, filename)
        submission_df.to_csv(path, index=False)
        print(f"Zapisano {path} – plik gotowy do wysyłki.")


if __name__ == "__main__":

    data_loader = DataLoader(
        "../input/dane-dla-dsc/sales_ads_train.csv",
        "../input/dane-dla-dsc/sales_ads_test.csv"
    )
    df = data_loader.load_data()

    preprocessor = Preprocessor(df)

    X = df[preprocessor.num_cols + preprocessor.cat_cols].copy()
    y = df["Cena"].copy()

    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    X_train_transformed = preprocessor.transform(X_train, y_train)
    X_val_transformed = preprocessor.transform(X_val)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 1000, 10000, step=1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
            "max_depth": trial.suggest_int("max_depth", 4, 12),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),
            "random_state": 42,
            "device": "cuda",          
            "tree_method": "hist",      
            "predictor": "gpu_predictor" 
        }

        model = xgb.XGBRegressor(**params)
        scores = cross_val_score(
            model, 
            X_train_transformed, 
            y_train, 
            scoring="neg_root_mean_squared_error", 
            cv=5
        )
        return -scores.mean()

    print("### Optymalizacja hiperparametrów za pomocą Optuny... ###")
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=20)

    best_params = study.best_params
    print("Najlepsze parametry znalezione przez Optunę:", best_params)

    best_params["random_state"] = 42
    best_params["device"] = "cuda"
    best_params["tree_method"] = "hist"
    best_params["predictor"] = "gpu_predictor"

    final_model_trainer = ModelTrainer(best_params)

    final_model_trainer.cross_validate(X_train_transformed, y_train)

    final_model_trainer.train(X_train_transformed, y_train)

    print("Wyniki na zbiorze walidacyjnym:")
    final_model_trainer.evaluate(X_val_transformed, y_val)

    feature_importance_df = final_model_trainer.feature_importance()
    feature_importance_df.to_csv("feature_importance_optuna.csv", index=False)
    print(feature_importance_df.head(20))

    test_df = data_loader.load_test_data()
    X_submission = test_df[preprocessor.num_cols + preprocessor.cat_cols].copy()
    X_submission_transformed = preprocessor.transform(X_submission)
    y_submission_pred = final_model_trainer.predict(X_submission_transformed)

    submission_handler = SubmissionHandler()
    submission_handler.save_submission(test_df, y_submission_pred)

Wynik był zadowalający, zbliżyliśmy się do przekroczenia 20 tysięcy RMSE, natomiast wciąż brakowało "tego czegoś". Wróciliśmy więc do podstaw i skupilśmy się na tych kolumnach które początkowo "droponeliśmy". Po ponownym przeanalzowaniiu kodu, zauważylismy, że "Wyposażenie" może odgrywać kluczową rolę. W najnowszej wersji dodaliśmy obsługę kolumny Wyposazenie, parsując ją na binarne cechy eq_* (np. eq_ABS), co wzbogaciło model o szczegółowe informacje o wyposażeniu pojazdów, przyczyniając się do obniżenia RMSE poniżej 20 tys., dokładnie do 19.5 tys. PLN. Zastąpiliśmy TargetEncoder bardziej zaawansowanym CatBoostEncoder, który lepiej radzi sobie z kodowaniem kategorycznym, poprawiając predykcję cen poprzez redukcję przecieku danych i lepsze uogólnienie. Rozszerzyliśmy zestaw cech kategorycznych o Naped, Skrzynia_biegow i Kraj_pochodzenia, a także ulepszyliśmy ModelTrainer, dodając mapowanie nazw cech w ważności, co ułatwiło analizę wpływu zmiennych na wynik. Naszym celem było uzyskanie najlepszego wyniku RMSE, co osiągnęliśmy dzięki tym zmianom, a dodatkowo postanowiliśmy rozważyć analizę reszt i skalowanie zmiennych numerycznych dla dalszej optymalizacji.

In [ ]:
import os
import ast
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import optuna

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from category_encoders import CatBoostEncoder

EXCHANGE_RATE_EUR_TO_PLN = 4.5 


def parse_equipment_inplace(df, col="Wyposazenie"):
    """
    Konwertuje kolumnę 'Wyposazenie' ze stringa typu "['ABS','CD']"
    na listę Pythona: ['ABS','CD'] (jeśli w ogóle kolumna istnieje).
    Dla brakujących wartości wstawiamy pustą listę.
    """
    if col not in df.columns:
        return  

    df[col] = df[col].fillna("[]")  
    def try_eval(x):
        try:
            if isinstance(x, str):
                return ast.literal_eval(x) 
            else:
                return x 
        except:
            return [] 

    df[col] = df[col].apply(try_eval)

def get_all_equipment_values(*dfs, col="Wyposazenie"):
    """
    Zwraca zbiór wszystkich elementów wyposażenia ze wskazanych DF-ów
    (np. train i test). Łączymy unikalne wartości w jeden set.
    """
    all_equip = set()
    for df in dfs:
        if col in df.columns:
            for val in df[col].dropna():
                if isinstance(val, list):
                    for item in val:
                        item = item.strip()  
                        all_equip.add(item)
    return all_equip

def add_equipment_features(df, all_equip, col="Wyposazenie"):
    """
    Dodaje do DF kolumny eq_XXX = 0/1, czy auto ma dany element wyposażenia.
    Wcześniej należy wywołać parse_equipment_inplace(df).
    """
    if col not in df.columns:
        return df

    for eq_item in all_equip:
        new_col = "eq_" + eq_item.replace(" ", "_").replace("/", "_")
        df[new_col] = 0

    for i, val in df[col].items():
        if isinstance(val, list):
            for item in val:
                item = item.strip()
                new_col = "eq_" + item.replace(" ", "_").replace("/", "_")
                if new_col in df.columns:
                    df.at[i, new_col] = 1

    return df

class DataLoader:
    def __init__(self, train_path, test_path):
        self.train_path = train_path
        self.test_path = test_path

    def load_data(self):
        print("Wczytuję dane z pliku CSV (train)...")
        df = pd.read_csv(self.train_path)

        df["Cena"] = df["Cena"].astype(float)

        df = self.convert_prices_to_pln(df)

        if "Waluta" in df.columns:
            df["Waluta"] = df["Waluta"].fillna("PLN")

        for col in [
            "Marka_pojazdu", 
            "Model_pojazdu", 
            "Wersja_pojazdu", 
            "Generacja_pojazdu",
            "Naped",
            "Skrzynia_biegow",
            "Kraj_pochodzenia",
        ]:
            if col in df.columns:
                df[col] = df[col].fillna("missing")

        parse_equipment_inplace(df, col="Wyposazenie")

        return df

    def load_test_data(self):
        print("Wczytuję dane testowe...")
        df = pd.read_csv(self.test_path)
        df["Is_EUR"] = df["Waluta"] == "EUR" 

        if "Waluta" in df.columns:
            df["Waluta"] = df["Waluta"].fillna("PLN")

        for col in [
            "Marka_pojazdu", 
            "Model_pojazdu", 
            "Wersja_pojazdu", 
            "Generacja_pojazdu",
            "Naped",
            "Skrzynia_biegow",
            "Kraj_pochodzenia",
        ]:
            if col in df.columns:
                df[col] = df[col].fillna("missing")

        parse_equipment_inplace(df, col="Wyposazenie")

        return df

    @staticmethod
    def convert_prices_to_pln(df):
        """Konwertuje ceny w EUR na PLN w zbiorze"""
        mask_eur = df["Waluta"] == "EUR"
        df["Cena"] = df["Cena"].astype(float)
        df.loc[mask_eur, "Cena"] = df.loc[mask_eur, "Cena"] * EXCHANGE_RATE_EUR_TO_PLN
        df["Waluta"] = "PLN"
        return df


class Preprocessor:
    def __init__(self, df):
        self.num_cols = [
            "Rok_produkcji",
            "Przebieg_km",
            "Moc_KM",
            "Pojemnosc_cm3",
            "Emisja_CO2",
            "Liczba_drzwi"
        ]
        
        self.cat_cols = [
            "Waluta",
            "Stan",
            "Marka_pojazdu",
            "Model_pojazdu",
            "Rodzaj_paliwa",
            "Typ_nadwozia",
            "Kolor",
            "Pierwszy_wlasciciel",
            "Wersja_pojazdu",
            "Generacja_pojazdu",
            "Naped",
            "Skrzynia_biegow",
            "Kraj_pochodzenia"
        ]
        
        eq_cols = [c for c in df.columns if c.startswith("eq_")]
        self.num_cols += eq_cols

        self.num_cols = [c for c in self.num_cols if c in df.columns]
        self.cat_cols = [c for c in self.cat_cols if c in df.columns]

        self.num_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])
        
        self.cat_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("catboost_enc", CatBoostEncoder())
        ])
        
        self.preprocessor = ColumnTransformer([
            ("num", self.num_pipeline, self.num_cols),
            ("cat", self.cat_pipeline, self.cat_cols)
        ])

        self.final_feature_names_ = self.num_cols + self.cat_cols

    def transform(self, X, y=None):
        if y is not None:
            return self.preprocessor.fit_transform(X, y)
        return self.preprocessor.transform(X)


class ModelTrainer:
    """
    Klasa do trenowania i ewaluacji modelu XGB w oryginalnej skali ceny (bez log).
    """
    def __init__(self, params, feature_names=None):
        self.model = xgb.XGBRegressor(**params)
        self.feature_names = feature_names 
        self.is_fitted = False
    
    def cross_validate(self, X, y):
        """
        Walidacja krzyżowa w oryginalnej skali (RMSE).
        """
        print("Przeprowadzam walidację krzyżową (oryginalna skala)...")
        cv_scores = cross_val_score(
            self.model, 
            X, 
            y, 
            scoring="neg_root_mean_squared_error", 
            cv=5
        )
        print(f"Średni RMSE (oryginalna skala) w walidacji krzyżowej: {-cv_scores.mean():.4f}")
    
    def train(self, X, y):
        """ Trenujemy model XGBRegressor (oryginalna skala targetu) """
        print("Trenuję model XGBRegressor (oryginalna skala targetu)...")
        self.model.fit(X, y)
        self.is_fitted = True
    
    def evaluate(self, X, y):
        """
        Ewaluacja w oryginalnej skali.
        """
        pred = self.model.predict(X)
        mae = mean_absolute_error(y, pred)
        rmse = np.sqrt(mean_squared_error(y, pred))
        r2 = r2_score(y, pred)
        print(f"[Oryg. skala] MAE = {mae:.3f}, RMSE = {rmse:.3f}, R2 = {r2:.3f}")
    
    def predict(self, X):
        """
        Zwracamy predykcje w oryginalnej skali.
        Dla submission przydaje się zaokrąglenie do integera (np. do PLN).
        """
        pred = self.model.predict(X)
        return np.round(pred).astype(int)
    
    def feature_importance(self, output_path="importance.csv"):
        """
        Zwraca DataFrame z importance feature'ów (wg GAIN) i dodatkowo zapisuje do CSV.
        Próbuje mapować "f0", "f1", ... do faktycznych nazw kolumn (self.feature_names).
        """
        if not self.is_fitted:
            print("Model nie jest jeszcze wytrenowany, brak importances.")
            return None
        
        booster = self.model.get_booster()
        importance_dict = booster.get_score(importance_type="gain")  
       
        items = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)

        feature_indices = []
        gains = []

        for k, v in items:
            idx_str = k.replace("f", "") 
            idx = int(idx_str)
            feature_indices.append(idx)
            gains.append(v)

        df_imp = pd.DataFrame({
            "FeatureIndex": feature_indices,
            "Gain": gains
        })

        if self.feature_names is not None and len(self.feature_names) > 0:
            df_imp["FeatureName"] = df_imp["FeatureIndex"].apply(
                lambda i: self.feature_names[i] if i < len(self.feature_names) else f"???_{i}"
            )
        else:
            df_imp["FeatureName"] = df_imp["FeatureIndex"].apply(lambda i: f"f{i}")

        df_imp = df_imp[["FeatureIndex", "FeatureName", "Gain"]].sort_values(by="Gain", ascending=False)

        df_imp.to_csv(output_path, index=False)
        print(f"Zapisano importance do pliku: {output_path}")
        return df_imp


class SubmissionHandler:
    """ Zapisywanie predykcji do pliku CSV. """
    def __init__(self, output_folder="tester"):
        os.makedirs(output_folder, exist_ok=True)
        self.output_folder = output_folder
    
    def save_submission(self, test_df, predictions, filename="submission.csv"):
        """
        Tutaj `predictions` są w PLN. Jeśli oryginalna waluta była w EUR,
        to konwertujemy z powrotem na EUR. 
        """
        test_df["Cena"] = predictions
        
        mask_eur = test_df["Is_EUR"]
        test_df.loc[mask_eur, "Cena"] = test_df.loc[mask_eur, "Cena"] / EXCHANGE_RATE_EUR_TO_PLN
        
        test_df["Cena"] = test_df["Cena"].round(2)
        
        submission_df = test_df[["ID", "Cena"]]
        path = os.path.join(self.output_folder, filename)
        submission_df.to_csv(path, index=False)
        print(f"Zapisano {path} – plik gotowy do wysyłki.")


if __name__ == "__main__":

    data_loader = DataLoader(
        "../input/dane-dla-dsc/sales_ads_train.csv",
        "../input/dane-dla-dsc/sales_ads_test.csv"
    )
    df_train = data_loader.load_data()
    df_test = data_loader.load_test_data()

    all_equip = get_all_equipment_values(df_train, df_test, col="Wyposazenie")

    df_train = add_equipment_features(df_train, all_equip, col="Wyposazenie")
    df_test = add_equipment_features(df_test, all_equip, col="Wyposazenie")

    if "Wyposazenie" in df_train.columns:
        df_train.drop(columns=["Wyposazenie"], inplace=True)
    if "Wyposazenie" in df_test.columns:
        df_test.drop(columns=["Wyposazenie"], inplace=True)

    preprocessor = Preprocessor(df_train)

    X = df_train[preprocessor.num_cols + preprocessor.cat_cols].copy()
    y = df_train["Cena"].values 

    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    X_train_transformed = preprocessor.transform(X_train, y_train)
    X_val_transformed = preprocessor.transform(X_val)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 1000, 10000, step=1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
            "max_depth": trial.suggest_int("max_depth", 4, 12),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),
            "random_state": 42,
            "device": "cuda",    
            "tree_method": "hist",
            "predictor": "gpu_predictor"
        }

        model = xgb.XGBRegressor(**params)
        cv_scores = cross_val_score(
            model,
            X_train_transformed,
            y_train,
            scoring="neg_root_mean_squared_error",
            cv=5
        )
        return -cv_scores.mean()  

    print("### Optymalizacja hiperparametrów za pomocą Optuny... ###")
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=10)

    best_params = study.best_params
    print("Najlepsze parametry znalezione przez Optunę:", best_params)

    best_params["random_state"] = 42
    best_params["device"] = "cuda"
    best_params["tree_method"] = "hist"
    best_params["predictor"] = "gpu_predictor"

  
    final_model_trainer = ModelTrainer(best_params, feature_names=preprocessor.final_feature_names_)
    final_model_trainer.cross_validate(X_train_transformed, y_train)

    final_model_trainer.train(X_train_transformed, y_train)

    print("Wyniki na zbiorze walidacyjnym (oryginalna skala):")
    final_model_trainer.evaluate(X_val_transformed, y_val)

    importance_df = final_model_trainer.feature_importance(output_path="importance.csv")
    print(importance_df.head(25))

    X_test = df_test[preprocessor.num_cols + preprocessor.cat_cols].copy()
    X_test_transformed = preprocessor.transform(X_test)
    y_test_pred = final_model_trainer.predict(X_test_transformed)

    submission_handler = SubmissionHandler()
    submission_handler.save_submission(df_test, y_test_pred, filename="submission.csv")


